<a href="https://colab.research.google.com/github/RaidahAhmed/NLP-GROUP-PROJECT-RAIDAH-AND-NAWE/blob/main/NLP_Group_Project_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Installing necessary dependencies
!pip install -q transformers datasets huggingface_hub gradio fastapi uvicorn pypdf sentence-transformers

# PART 2: PUTTING THE SENTIMENT MODEL ONLINE

# 2.1 Logging into huggingface

In [2]:
#Logging into huggingface
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login, whoami
login(token=HF_TOKEN)
HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: RaidahAhmed


# 2.2 Loading the model

In [3]:
#Loading model from drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#Loading the model
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

LOCAL_MODEL_DIR = "/content/drive/MyDrive/my_model"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(LOCAL_MODEL_DIR)

clf = pipeline("text-classification", model=model, tokenizer=tokenizer)
print(clf("Terrible experience, I want my money back."))
print(clf("I really enjoyed this class!"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9976874589920044}]
[{'label': 'POSITIVE', 'score': 0.995557963848114}]


# 2.3 Uploading model to hugging face

In [5]:
MODEL_NAME = "RaidahandNawe-sentiment-model"
MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}"

model.push_to_hub(MODEL_ID)
tokenizer.push_to_hub(MODEL_ID)
print("Model page:", f"https://huggingface.co/{MODEL_ID}")

README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...bbp2y11/model.safetensors:   9%|8         | 24.0MB /  268MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


Model page: https://huggingface.co/RaidahAhmed/RaidahandNawe-sentiment-model


# 2.4 Writing the model card

In [6]:
from huggingface_hub import ModelCard

ACC = 0.91
F1 = 0.9106


example_sentences = [
    "Terrible experience, I want my money back.",
    "I really enjoyed this class!",
    "What was this all about, very confusing!",
]
EXAMPLES = "\n".join(
    f'- "{s}" -> {clf(s)[0]["label"]} (score {clf(s)[0]["score"]:.2f})' for s in example_sentences
)

LIMITATIONS = ("Trained only on English movie reviews, so it may do worse on other topics. "
               "It can be wrong on sarcasm, mixed opinions, and mixed-language text. "
               "Reviews longer than 256 tokens are cut.")

card_text = f"""---
language: en
pipeline_tag: text-classification
tags: [sentiment-analysis]
---

# {MODEL_NAME}

Sentiment classifier (positive / negative) fine-tuned from `distilbert-base-uncased`.

**Group members:** Raidah and Nawe
**Dataset:** stanfordnlp/imdb (movie reviews)

## Results on the test set
| Metric | Score |
|---|---|
| Accuracy | {ACC} |
| F1 | {F1} |

## Example predictions
{EXAMPLES}

## Limitations
{LIMITATIONS}
"""

ModelCard(card_text).push_to_hub(MODEL_ID)
print("Model card uploaded.")

No files have been modified since last commit. Skipping to prevent empty commit.


Model card uploaded.


# 2.5 Loading the model back from the internet

In [7]:
hub_clf = pipeline("text-classification", model=MODEL_ID)
hub_clf("I really enjoyed this class!")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.995557963848114}]

# 2.6 Trying the model using Gradio

In [8]:
import gradio as gr

def predict(text):
    if not text.strip():
        return {}
    result = hub_clf(text)[0]
    score = result["score"]
    other = "NEGATIVE" if result["label"] == "POSITIVE" else "POSITIVE"
    return {result["label"]: score, other: 1 - score}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    description="Type a sentence and see if it is positive or negative.",
    examples=["The food was amazing!", "Worst service ever.", "It was okay, nothing special."],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb2bcd9d9893d0b50c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 2.7 Save the demo code

In [9]:
print(MODEL_ID)

RaidahAhmed/RaidahandNawe-sentiment-model


# 2.8 Writing app.py file

In [10]:
%%writefile app.py
import gradio as gr
from transformers import pipeline

MODEL_ID = "RaidahAhmed/RaidahandNawe-sentiment-model"
clf = pipeline("text-classification", model=MODEL_ID)

def predict(text):
    if not text.strip():
        return {}
    r = clf(text)[0]
    other = "NEGATIVE" if r["label"] == "POSITIVE" else "POSITIVE"
    return {r["label"]: r["score"], other: 1 - r["score"]}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    examples=["The food was amazing!", "Worst service ever."],
)
demo.launch()

Writing app.py


In [11]:
%%writefile requirements.txt
transformers
torch
gradio
fastapi
uvicorn

Writing requirements.txt


## 2.9 · Make an API with FastAPI

In [12]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline

MODEL_ID = "RaidahAhmed/RaidahandNawe-sentiment-model"   # paste your model id

app = FastAPI(title="Sentiment API")

# Load the model ONCE when the app starts, not on every request (that would be very slow)
clf = pipeline("text-classification", model=MODEL_ID)

class TextIn(BaseModel):
    text: str

class PredictionOut(BaseModel):
    label: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionOut)
def predict(item: TextIn):
    if not item.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    result = clf(item.text)[0]
    return PredictionOut(label=result["label"], score=round(result["score"], 4))

Writing main.py


In [13]:
#Starting the API
import subprocess, time
server = subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(25)
print("API started")

API started


In [14]:
#Testing the API
import requests

BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/health").json())

for sentence in ["I love this app!", "This is so frustrating.", ""]:
    r = requests.post(f"{BASE}/predict", json={"text": sentence})
    print(r.status_code, r.json())

{'status': 'ok'}
200 {'label': 'POSITIVE', 'score': 0.9968}
200 {'label': 'NEGATIVE', 'score': 0.9954}
400 {'detail': 'Text cannot be empty'}


In [15]:
#Stopping the API so it does not keep running in the background
server.terminate()
print("API stopped")

API stopped


# PART 3: BUILDING AN LLM CHATBOT

In [25]:
from transformers import pipeline

# Load a small chat model that runs inside Colab
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
pipe = pipeline(
    "text-generation",
    model=LLM_MODEL,
    device_map="auto",       # uses the GPU if there is one, otherwise the CPU
    torch_dtype="auto",
)

def ask_llm(messages, max_tokens=400):
    # The pipeline accepts the same list of {"role", "content"} messages
    out = pipe(messages, max_new_tokens=max_tokens, do_sample=False)   # do_sample=False = steady, repeatable answers
    # The result holds the whole conversation; the last message is the model's answer
    return out[0]["generated_text"][-1]["content"]

print(ask_llm([
    {"role": "system", "content": "You are a helpful assistant. Answer in one sentence."},
    {"role": "user", "content": "What is natural language processing?"},
]))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Natural Language Processing (NLP) is the branch of artificial intelligence that enables computers to understand, interpret, and generate human language.


#Comparing the LLM and my Fine-tuned model

In [26]:
# It gives the SAME 5 sentences to two different models(judges) and compares them:
#   1. mine = my fine-tuned DistilBERT model (trained for this)
#   2. llm  = the LLM, which gets no training, only a prompt telling it what to do

import time, json   # time = to measure seconds

# The 5 test sentences
test_sentences = [
    "The delivery was quick and the packaging was great.",
    "I waited two hours and nobody helped me.",
    "Not bad, but I expected more for the price.",
    "Absolutely brilliant lecture today!",
    "The app keeps crashing every time I open it.",
]

for s in test_sentences:
    # My model
    t0 = time.time()                                   # start the timer
    mine = hub_clf(s)[0]["label"]                      # my model's answer: POSITIVE or NEGATIVE
    t_mine = time.time() - t0                          # stop the timer

    # Judge 2: the LLM
    t0 = time.time()
    llm_label = ask_llm([
        # The "system" message is the rule we give the LLM: reply with ONE word only
        {"role": "system", "content": "Classify the sentiment. Reply with ONE word: POSITIVE or NEGATIVE."},
        # The "user" message is the sentence we want it to judge
        {"role": "user", "content": s},
    ], max_tokens=5).strip()
    t_llm = time.time() - t0

    # Print one line per sentence: the sentence, both answers, and both times
    print(f"{s[:45]:<47} mine={mine:<9} ({t_mine:.2f}s)  llm={llm_label:<9} ({t_llm:.2f}s)")

The delivery was quick and the packaging was    mine=POSITIVE  (1.89s)  llm=POSITIVE  (13.45s)
I waited two hours and nobody helped me.        mine=NEGATIVE  (0.07s)  llm=NEGATIVE  (10.58s)
Not bad, but I expected more for the price.     mine=POSITIVE  (0.05s)  llm=NEGATIVE  (11.04s)
Absolutely brilliant lecture today!             mine=POSITIVE  (0.06s)  llm=POSITIVE  (9.55s)
The app keeps crashing every time I open it.    mine=NEGATIVE  (0.05s)  llm=NEGATIVE  (10.76s)


#Getting JSON from an LLM

In [27]:
complaint = "I paid for MTN data bundles yesterday but they never came. This is the third time! Fix it today."

reply = ask_llm([
    {"role": "system", "content": (
        "Extract information from customer complaints. "
        "Reply ONLY with JSON with keys: product, issue, urgency (low, medium or high). No other text."
    )},
    {"role": "user", "content": complaint},
])
print(reply)
data = json.loads(reply.strip().strip("`").removeprefix("json"))
data

{
  "product": "MTN data bundles",
  "issue": "Delivery of purchased products",
  "urgency": "high"
}


{'product': 'MTN data bundles',
 'issue': 'Delivery of purchased products',
 'urgency': 'high'}

#Answering questions from a PDF

In [28]:
# Uploading PDF from my computer into Colab.
from google.colab import files
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print("Using:", PDF_PATH)

Saving Understanding-endometriosis- pack_0.pdf to Understanding-endometriosis- pack_0.pdf
Using: Understanding-endometriosis- pack_0.pdf


In [29]:
#Reading the text of every page and remember each page's number, later the bot will tell us which page an answer came from.
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]   # a list of (page number, text)
print(f"{len(pages)} pages. First 300 characters of page 1:\n")
print(pages[0][1][:300])

5 pages. First 300 characters of page 1:

 
 
Understanding Endometriosis - Information Pack 
 
What is endometriosis? 
 
Endometriosis (pronounced en- doh – mee – tree – oh – sis) is the name given to the condition where 
cells like the ones in the lining of the womb (uterus) are found elsewhere in the body. 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 



In [30]:
# Cutting every page into small pieces (chunks) of about 500 characters.
#  Because a whole PDF is too big to hand to the model. Small chunks let us find the exact part we need.
def chunk_pages(pages, chunk_size=500, overlap=100):
    chunks = []
    for page_num, text in pages:
        text = " ".join(text.split())          # cleans up extra spaces and line breaks
        start = 0
        while start < len(text):
            piece = text[start:start + chunk_size]
            if len(piece) > 50:                # skips tiny leftovers
                chunks.append({"page": page_num, "text": piece})   # keep the page number with the text
            start += chunk_size - overlap      # move forward, but overlap by 100 so sentences aren't cut in half
    return chunks

chunks = chunk_pages(pages)
print(len(chunks), "chunks")
print(chunks[0])                               # look at the first chunk

31 chunks
{'page': 1, 'text': 'Understanding Endometriosis - Information Pack What is endometriosis? Endometriosis (pronounced en- doh – mee – tree – oh – sis) is the name given to the condition where cells like the ones in the lining of the womb (uterus) are found elsewhere in the body. • Endometriosis is not an infection. • Endometriosis is not contagious. • Endometriosis is not cancer. Every month a woman’s body goes through hormonal changes. Hormones are naturally released which cause the lining of the womb to increase in'}


In [31]:
# Turning every chunk into embeddingS that capture its meaning.
# Chunks with similar meaning get similar numbers, so we can search by meaning, not just by exact words.
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(chunk_vectors.shape)   # (number of chunks, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(31, 384)


In [32]:
# Retrive most relevant chunks for a quaestion;given a question, find the 3 chunks whose meaning is closest to it.
def retrieve(question, k=3):
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]   # turns the question into numbers too
    scores = chunk_vectors @ q_vec            # similarity score for every chunk (higher = closer meaning)
    top = np.argsort(scores)[::-1][:k]        # positions of the k best scores
    return [{**chunks[i], "score": float(scores[i])} for i in top]

question = "Why does endometriosis occur?"
for hit in retrieve(question):
    print(f"page {hit['page']}  score={hit['score']:.2f}  {hit['text'][:120]}...")

page 1  score=0.75  Understanding Endometriosis - Information Pack What is endometriosis? Endometriosis (pronounced en- doh – mee – tree – o...
page 2  score=0.75  he bloodstream. This could explain why it has been found in areas such as the eyes and brain. • Genetic predisposition t...
page 1  score=0.73  ed adenomyosis). Endometriosis can grow in existing scars from previous operations. In rare cases, it has been found in ...


In [33]:
#Let LLM answer from these chuncks only
#The "system prompt" is the rulebook we give the LLM.
SYSTEM_PROMPT = (
    "You answer questions using ONLY the context provided. "
    "Cite page numbers like (page 3). "
    "If the answer is not in the context, say: I don't know, that is not in the document."
)

def rag_answer(question, k=3):
    hits = retrieve(question, k)                                                    # 1. find the best chunks
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)           # 2. join them into one block of text
    return ask_llm([                                                                 # 3. ask the LLM, using only that block
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])

print(rag_answer(question))

There is no proven cause for endometriosis, but there are several theories about what might trigger its occurrence. These include genetic predisposition, where the condition runs in families due to inherited traits; immune dysfunction, suggesting that the body's immune system struggles to eliminate endometrial tissue; and environmental factors or lifestyle choices that may contribute to its development. Each theory attempts to explain different aspects of endometriosis, but none fully explains all cases.


#Testing the LLM properly

In [34]:
# Will ask 5 questions: 3 your PDF can answer and 2 it cannot.
test_questions = [
    "What is endometriosis?",
    "How common is endometriosis?",
    "Give me ten symptoms of endometriosis.",
    "Who won the 2026 World Cup?",                  # NOT in the document
    "Where is WITU located",                        # NOT in the document
]
for q in test_questions:
    print("Q:", q)
    top = retrieve(q, 1)[0]
    print(f"   (best page found: {top['page']})")   # fills the "Right page found?" column
    print("A:", rag_answer(q), "\n")

Q: What is endometriosis?
   (best page found: 1)
A: Endometriosis is a condition where cells similar to those in the uterus's lining are found outside the uterus in various parts of the body. It is not an infection, contagious, or cancerous. The condition occurs due to hormonal changes during menstruation when hormones trigger the growth of these uterine-like cells in different locations within the body. 

Q: How common is endometriosis?
   (best page found: 4)
A: Around 1.5 - 2 million women in the UK have endometriosis. 

Q: Give me ten symptoms of endometriosis.
   (best page found: 3)
A: Painful periods, painful sex, infertility, depression, frequent infections such as thrush, feeling faint/fainting during periods, leg pain, back pain, bowel and bladder symptoms including painful bowel movements, heavy periods with clots, prolonged bleeding, premenstrual spotting, irregular periods, loss of dark or old blood before a period or at the end of a period. 

Q: Who won the 2026 World Cu

#Turning the LLM into a chatbot

In [41]:
import gradio as gr

# Gradio sends the chat history with extra fields.
def clean_history(history):
    out = []
    for m in history[-6:]:                          # last few turns only, to keep it short and fast
        content = m["content"]
        if isinstance(content, list):
            content = " ".join(p.get("text", "") for p in content if isinstance(p, dict))
        out.append({"role": m["role"], "content": content})
    return out

def chat(message, history):
    hits = retrieve(message)                                                   # find the best chunks for the new message
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)

    # Rulebook
    rules = (SYSTEM_PROMPT
             + " Be polite. Refuse questions unrelated to the document."
             + " You are not a doctor: for personal medical concerns, tell the person to see a healthcare professional."
             + " If the answer is not in the context, reply exactly: I don't know, that is not in the document.")

    messages = [{"role": "system", "content": rules}]
    messages += clean_history(history)

    messages.append({"role": "user", "content": (
        f"Context:\n{context}\n\n"
        f"Question: {message}\n\n"
        "Reminder: answer ONLY from the context and END with the page number in brackets OR page numbers if many for example (page 3). "
    )})
    return ask_llm(messages)

BOT_TITLE = "Endometriosis Info Assistant"

bot = gr.ChatInterface(fn=chat, title=BOT_TITLE,
                           examples=["What is this document about?"])
bot.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e4a6ae61ec95f5d129.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [42]:
print(PDF_PATH)

Understanding-endometriosis- pack_0.pdf


In [44]:
%%writefile chatbot_app.py
import re
import numpy as np
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

PDF_PATH = "Understanding-endometriosis- pack_0.pdf"
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_PROMPT = (
    " Be polite. Refuse questions unrelated to the document."
    " You are not a doctor: for personal medical concerns, tell the person to see a healthcare professional."
    " If the answer is not in the context, reply exactly: I don't know, that is not in the document.")

# The LLM runs on this computer
pipe = pipeline("text-generation", model=LLM_MODEL, device_map="auto", torch_dtype="auto")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Read the PDF and keep each page's number
pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(PdfReader(PDF_PATH).pages)]

# Split into chunks of about 500 characters, overlapping by 100
chunks = []
for page_num, text in pages:
    text = " ".join(text.split())
    for start in range(0, len(text), 400):
        piece = text[start:start + 500]
        if len(piece) > 50:
            chunks.append({"page": page_num, "text": piece})
vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)

def retrieve(q, k=3):
    scores = vectors @ embedder.encode([q], normalize_embeddings=True)[0]
    return [chunks[i] for i in np.argsort(scores)[::-1][:k]]

def clean_history(history):
    out = []
    for m in history[-6:]:
        content = m["content"]
        if isinstance(content, list):
            content = " ".join(p.get("text", "") for p in content if isinstance(p, dict))
        out.append({"role": m["role"], "content": content})
    return out

def chat(message, history):
    hits = retrieve(message)
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + clean_history(history)
    messages.append({"role": "user", "content": (
        f"Context:\n{context}\n\n"
        f"Question: {message}\n\n"
        "Reminder: answer ONLY from the context and END with the page number in brackets OR page numbers if many for example (page 3)."
    )})
    out = pipe(messages, max_new_tokens=400, do_sample=False)
    answer = out[0]["generated_text"][-1]["content"]
    if "don't know" not in answer.lower() and not re.search(r"page\s*\d+", answer, re.IGNORECASE):
        answer += f" (page {hits[0]['page']})"
    return answer

    gr.ChatInterface(fn=chat, title="Endometriosis Info Assistant").launch()

Overwriting chatbot_app.py


In [45]:
%%writefile chatbot_requirements.txt
pypdf
sentence-transformers
transformers
torch
accelerate
numpy
gradio

Overwriting chatbot_requirements.txt
